### One Hot Encoding

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("./cars.csv")
df.head(4)

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000


In [43]:
brands_cnt = df["brand"].value_counts()
brands_cnt

brand
Maruti           2448
Hyundai          1415
Mahindra          772
Tata              734
Toyota            488
Honda             467
Ford              397
Chevrolet         230
Renault           228
Volkswagen        186
BMW               120
Skoda             105
Nissan             81
Jaguar             71
Volvo              67
Datsun             65
Mercedes-Benz      54
Fiat               47
Audi               40
Lexus              34
Jeep               31
Mitsubishi         14
Land                6
Force               6
Isuzu               5
Ambassador          4
Kia                 4
MG                  3
Daewoo              3
Ashok               1
Opel                1
Peugeot             1
Name: count, dtype: int64

1. Onehot using pandas

In [8]:
#apply on fuel and owner bcz both has less no of values
print(df["fuel"].unique(), df["owner"].unique())

['Diesel' 'Petrol' 'LPG' 'CNG'] ['First Owner' 'Second Owner' 'Third Owner' 'Fourth & Above Owner'
 'Test Drive Car']


In [ ]:
#here columns denote which have to do encode
pd.get_dummies(df, columns=["fuel", "owner"])

,brand,km_driven,selling_price,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,False,True,False,False,True,False,False,False,False
1,Skoda,120000,370000,False,True,False,False,False,False,True,False,False
2,Honda,140000,158000,False,False,False,True,False,False,False,False,True
3,Hyundai,127000,225000,False,True,False,False,True,False,False,False,False
4,Maruti,120000,130000,False,False,False,True,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,False,False,False,True,True,False,False,False,False
8124,Hyundai,119000,135000,False,True,False,False,False,True,False,False,False
8125,Maruti,120000,382000,False,True,False,False,True,False,False,False,False
8126,Tata,25000,290000,False,True,False,False,True,False,False,False,False


2. K-1 encdoe 
bcz we need to remove 1 col to each category

In [10]:
#here columns denote which have to do encode
pd.get_dummies(df, columns=["fuel", "owner"], drop_first=True)
#here via drop-first first col is remove

,brand,km_driven,selling_price,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,True,False,False,False,False,False,False
1,Skoda,120000,370000,True,False,False,False,True,False,False
2,Honda,140000,158000,False,False,True,False,False,False,True
3,Hyundai,127000,225000,True,False,False,False,False,False,False
4,Maruti,120000,130000,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,False,False,True,False,False,False,False
8124,Hyundai,119000,135000,True,False,False,True,False,False,False
8125,Maruti,120000,382000,True,False,False,False,False,False,False
8126,Tata,25000,290000,True,False,False,False,False,False,False


3. Using sklearn - best option

bcz pd not remeber of position of encode data every tie get differnt - this problem solved via sklean

In [11]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

In [13]:
X = df.iloc[:, :4]
Y = df.iloc[:, 4:]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.33, random_state=45)

In [25]:
ohe =  OneHotEncoder(sparse_output=False, drop="first",dtype=np.int32)
#via sparse_output return sparse matrix but need np-array, drop = first => remove first col

ohe.fit(X_train[["fuel", "owner"]])

X_train_encode = ohe.transform(X_train[["fuel", "owner"]])
X_test_encode = ohe.transform(X_test[["fuel", "owner"]])

X_train_encode

array([[0, 0, 1, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       ...,
       [1, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 1, 0, 0]], shape=(5445, 7), dtype=int32)

In [29]:
narr = np.hstack((X_train[["brand", "km_driven"]], X_train_encode))
narr

array([['Chevrolet', 80000, 0, ..., 0, 0, 0],
       ['Maruti', 58343, 0, ..., 0, 0, 0],
       ['Hyundai', 30000, 0, ..., 0, 0, 0],
       ...,
       ['Hyundai', 76460, 1, ..., 0, 0, 0],
       ['Volvo', 20000, 1, ..., 0, 0, 0],
       ['Tata', 120000, 1, ..., 1, 0, 0]], shape=(5445, 9), dtype=object)

In [45]:
#now encode the brand

thresold = 100 #means if brand count is less than 100 make other 

other = (brands_cnt[(brands_cnt)<= thresold]).index #get all names
other

Index(['Nissan', 'Jaguar', 'Volvo', 'Datsun', 'Mercedes-Benz', 'Fiat', 'Audi',
       'Lexus', 'Jeep', 'Mitsubishi', 'Land', 'Force', 'Isuzu', 'Ambassador',
       'Kia', 'MG', 'Daewoo', 'Ashok', 'Opel', 'Peugeot'],
      dtype='object', name='brand')

In [48]:
pd.get_dummies(df["brand"].replace(other, "other"), dtype=np.int16, drop_first=True)

,Chevrolet,Ford,Honda,Hyundai,Mahindra,Maruti,Renault,Skoda,Tata,Toyota,Volkswagen,other
0,0,0,0,0,0,1,0,0,0,0,0,0
1,0,0,0,0,0,0,0,1,0,0,0,0
2,0,0,1,0,0,0,0,0,0,0,0,0
3,0,0,0,1,0,0,0,0,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
8123,0,0,0,1,0,0,0,0,0,0,0,0
8124,0,0,0,1,0,0,0,0,0,0,0,0
8125,0,0,0,0,0,1,0,0,0,0,0,0
8126,0,0,0,0,0,0,0,0,1,0,0,0
